# Galerie modèles pré-entraînés — [03 🟠] Détecter les questions en doublon (support FAQ)

> Optionnel. **À COMPLÉTER** : des `# TODO`. Tu reconstruis le code en t'appuyant sur les notebooks 01-02 et `panorama_huggingface_hub.md` — et si tu bloques, une **solution repliée** t'attend en fin de notebook (essaie d'abord sans). Le guidage **décroît** au fil du notebook.

**Contexte fictif.** Un service support reçoit des questions souvent **formulées différemment mais identiques sur le fond**. On veut **repérer les doublons** pour les regrouper. Outil : les **embeddings** (un texte → un vecteur) + la **similarité cosinus**.

Pattern : 1) choisir le modèle d'embeddings → 2) encoder → 3) calculer les similarités → 4) seuiller pour décider « doublon ou pas ».


## Setup

> ⚠️ **Premier lancement = téléchargement du modèle** (connexion requise une fois, puis mise en cache locale). Tout tourne sur **CPU**, pas besoin de GPU.

```bash
pip install "transformers>=4.40" torch sentence-transformers scikit-learn pandas
```

> 💡 Pour les embeddings on utilise la librairie **`sentence-transformers`** (et non `pipeline()`).


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## [1] Choisir le modèle d'embeddings (guidé)

- **Modèle** : `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
- **Tâche** : embeddings de phrases, multilingue (français ✅)
- **Taille** : ~470 Mo · dimension des vecteurs : 384


In [ ]:
MODELE = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
emb = SentenceTransformer(MODELE)   # fourni
print("dimension :", emb.get_sentence_embedding_dimension())

## [2] Encoder les questions (à compléter)

Voici les questions du support. **TODO** : encode-les en vecteurs avec `emb.encode(...)`.

In [ ]:
questions = [
    "Comment réinitialiser mon mot de passe ?",
    "J'ai oublié mes identifiants de connexion",        # doublon de la 1re ?
    "Quels sont les horaires du support ?",
    "À quelle heure puis-je vous joindre ?",             # doublon de la 3e ?
]

# TODO : vecteurs = emb.encode(questions)
vecteurs = ...  # à compléter

## [3] Calculer la matrice de similarité (à compléter)

**TODO** : calcule `cosine_similarity(vecteurs)` et affiche la matrice. Plus la valeur est proche de 1, plus les deux questions sont proches sémantiquement.

In [ ]:
# TODO : sim = cosine_similarity(vecteurs)
# TODO : affiche sim (par ex. avec pandas pour la lisibilité)


## [4] Décider « doublon » par seuillage (peu guidé)

**TODO** : parcours les paires de questions distinctes et signale comme **doublon** celles dont la similarité dépasse un seuil que tu choisis. Affiche les paires détectées.

*Indice : commence par `0.6`… et ne t'étonne pas si **rien ne ressort**. Regarde ta matrice : les deux vraies paires — (0,1) et (2,3) — se détachent nettement des autres, mais **en dessous de 0.6**. À toi de choisir un seuil qui sépare les deux blocs. C'est la leçon du notebook : un seuil ne se décrète pas, il se lit dans les données.*

In [ ]:
# TODO : à toi de jouer


## ✅ Résultats attendus (ton filet d'auto-vérification)

Avec `paraphrase-multilingual-MiniLM-L12-v2` (dimension 384), ta matrice de
similarité doit ressembler à ceci (±0.05 selon les versions des librairies) :

| | Q0 mot de passe | Q1 identifiants | Q2 horaires | Q3 quelle heure |
|---|---|---|---|---|
| **Q0** | 1.00 | **0.37** | 0.04 | 0.09 |
| **Q1** | **0.37** | 1.00 | 0.09 | 0.16 |
| **Q2** | 0.04 | 0.09 | 1.00 | **0.54** |
| **Q3** | 0.09 | 0.16 | **0.54** | 1.00 |

Ce qu'il faut vérifier :

- les deux paires de doublons — **(0,1) à ~0.37** et **(2,3) à ~0.54** — sont
  nettement au-dessus de toutes les paires croisées (**≤ 0.16**) ;
- un seuil autour de **0.3** sépare donc proprement doublons et non-doublons ;
- avec le seuil naïf de 0.6, **aucun doublon ne ressort** — c'est normal, et
  c'est exactement pourquoi on ne fixe jamais un seuil sans regarder les
  similarités réelles de SES données.

Si tes valeurs sont très différentes (par ex. tout > 0.9 ou tout < 0.05),
vérifie que tu as bien encodé les **4 questions** (shape `(4, 384)`) et que tu
compares les vecteurs entre eux (`cosine_similarity(vecteurs)`).


## 🔓 Solution repliée (à n'ouvrir qu'après avoir essayé)

<details>
<summary>👉 Cliquer pour afficher la solution complète</summary>

```python
# [2] Encoder les questions
vecteurs = emb.encode(questions)
print(vecteurs.shape)          # (4, 384)

# [3] Matrice de similarité
import pandas as pd
sim = cosine_similarity(vecteurs)
print(pd.DataFrame(sim).round(2))

# [4] Détection des doublons par seuillage
SEUIL = 0.3   # lu dans la matrice : les doublons sont à 0.37 et 0.54,
              # les paires croisées à 0.16 max — 0.6 ne détecterait rien
for i in range(len(questions)):
    for j in range(i + 1, len(questions)):
        if sim[i, j] >= SEUIL:
            print(f"DOUBLON ({i},{j}) sim={sim[i, j]:.2f}")
            print(f"   - {questions[i]}")
            print(f"   - {questions[j]}")
```

Sortie attendue : les paires **(0,1)** et **(2,3)**, et rien d'autre.

</details>


## 🤔 Question réflexive — le choix du seuil

Le **seuil** est un arbitrage : trop bas → faux doublons (on fusionne des questions différentes) ; trop haut → on rate des doublons. 

> 💡 C'est le **même geste de seuil** que partout dans le parcours (seuil de rejet M7, seuil de revue humaine M8) : un curseur métier, pas une vérité.

**Sobriété** : MiniLM (~470 Mo) suffit largement ici. Inutile de charger un gros modèle d'embeddings — vérifie toi-même que la séparation doublon/non-doublon est déjà nette.